# LLM Story Evaluation - Evaluation Template

**Course:** IT469 — Human Language Technologies  
**Section:** 62723  
**Group:** 2

**Team Members:**
- Nouf AlMansour (444200525)
- Tarfah Bin Moammar (444200611)
- Shahad AlMutairi (444200935)
- Nour AlGhomlas (444200811)
- Lujain AlHarbi (443200811)

**Supervised by:** Dr. Hend AlRasheed

---

## Purpose of This Notebook

This is the **template evaluation notebook** used by all team members to generate LLM evaluations for the IT469 project on prompting techniques for story assessment.

**What this notebook does:**
- Loads the HANNA benchmark dataset (1,056 human-annotated stories)
- Implements 5 prompting strategies: Role 1 (Human Annotator), Role 2 (Story Expert), Role 3 (Creative Writing Professor), Instruction-Based, Zero-Shot
- Evaluates stories using GPT-4.1 Mini or Claude variants (Sonnet 4.5, Haiku 4.5)
- Outputs evaluation results as CSV files for analysis

**How it was used:**
Each team member ran this notebook with different configurations to collect evaluations across all model-technique combinations:
- GPT-4.1 Mini: 5 techniques (Role 1, Role 2, Role 3, Instruction-Based, Zero-Shot)
- Claude Sonnet 4.5: 4 techniques (Role 1, Role 2, Role 3, Zero-Shot)
- Claude Haiku 4.5: 1 technique (Instruction-Based)

**Output:**
- CSV files containing LLM scores for each story across 6 narrative dimensions (Relevance, Coherence, Empathy, Surprise, Engagement, Complexity)
- Each story evaluated 3 times per configuration to reduce stochastic variation


---

## How to Use This Template

1. **Set API keys** - Add your OpenAI or Anthropic API key in the configuration section
2. **Select model** - Choose GPT-4.1 Mini or Claude variant
3. **Select technique** - Choose one of the 5 prompting strategies
4. **Run evaluation** - Execute all cells to generate evaluations
5. **Save output** - Results saved as CSV for downstream analysis

> **Note:** This notebook generates evaluation data. For analysis and visualization of results, see the separate analysis notebook.

---

### Install packages

In [ ]:
# Install required packages
!pip install -q openai google-genai anthropic pandas numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.5/627.5 kB 14.8 MB/s eta 0:00:00


### Imports

In [ ]:
# Standard library imports
import os
import json
import time
import re
from getpass import getpass
from statistics import mean

# Third-party imports
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# Model SDK imports
from openai import OpenAI
from google import genai
from anthropic import Anthropic

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### configuration

In [ ]:
# Experiment configuration
PROMPT_VERSION = "technique_prompting" #change this for your prompt technique
REPEATS_PER_MODEL = 3

# Dataset configuration
DATASET_FILE = "stories_with_avg_human_scores.csv"
USE_SAMPLE = False #True if you'd like to test a sample first
SAMPLE_SIZE = 10
SAMPLE_RANDOM_STATE = 42

# Output configuration
SAVE_EVERY_N_STORIES = 5
OUTPUT_DIR = "results"

# Model names
GPT_MODEL = "gpt-4.1-mini"
CLAUDE_MODEL = "claude-3-haiku-20240307"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/ModelEvaluationResults" # Change this to your desired Google Drive path
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

### API keys and clients

In [ ]:
# Load API keys securely
os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key: ")
os.environ["ANTHROPIC_API_KEY"] = getpass("Enter Anthropic API key: ")

# Initialize clients
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
anthropic_client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

Enter OpenAI API key: ··········
Enter Gemini API key: ··········
Enter Anthropic API key: ··········


### Load dataset

In [ ]:
# Load the preprocessed dataset
gold_df = pd.read_csv(DATASET_FILE)

# Show basic information
print("Dataset shape:", gold_df.shape)
print("Columns:")
print(gold_df.columns.tolist())

gold_df.head()

Dataset shape: (1056, 13)
Columns:
['story_id', 'prompt', 'human_reference_story', 'story', 'source_model', 'avg_relevance', 'avg_coherence', 'avg_empathy', 'avg_surprise', 'avg_engagement', 'avg_complexity', 'num_ratings', 'story_type']


,story_id,prompt,human_reference_story,story,source_model,avg_relevance,avg_coherence,avg_empathy,avg_surprise,avg_engagement,avg_complexity,num_ratings,story_type
0,0,When you die the afterlife is an arena where y...,"3,000 years have I been fighting. Every mornin...","3,000 years have I been fighting. Every mornin...",Human,3.666667,3.666667,2.333333,2.333333,3.333333,2.666667,3,human
1,1,A new law is enacted that erases soldiers memo...,"“Dad, you 're on TV again !” I heard Eric 's v...","“Dad, you 're on TV again !” I heard Eric 's v...",Human,5.000000,4.666667,4.000000,3.666667,3.666667,4.000000,3,human
2,2,A scientific study proves that all humans have...,"When Tyler entered the ward, his daughter Vale...","When Tyler entered the ward, his daughter Vale...",Human,4.666667,4.666667,4.000000,4.333333,4.000000,4.333333,3,human
3,3,Write a story about an elderly wizard and his ...,His body was failing. He had taken care of it ...,His body was failing. He had taken care of it ...,Human,3.666667,4.000000,3.000000,2.000000,3.666667,4.000000,3,human
4,4,"You have become death, destroyer of worlds.","I saw the button. It was simple, red, no words...","I saw the button. It was simple, red, no words...",Human,4.666667,5.000000,3.000000,4.666667,3.666667,3.666667,3,human


### evaluation dataframe

In [ ]:
# Use either the full dataset or a debug sample
if USE_SAMPLE:
    eval_df = gold_df.sample(SAMPLE_SIZE, random_state=SAMPLE_RANDOM_STATE).copy()
    print(f"Using sample of {len(eval_df)} stories")
else:
    eval_df = gold_df.copy()
    print(f"Using full dataset of {len(eval_df)} stories")

# Reset index for clean looping
eval_df = eval_df.reset_index(drop=True)

# Preview selected data
eval_df[["story_id", "source_model", "prompt"]].head()

Using sample of 10 stories


,story_id,source_model,prompt
0,260,CTRL,"A 4-D star collides with Earth, causing it to ..."
1,832,Fusion,This picture gets more horrific the longer I l...
2,846,Fusion,A hardcore doomsday prepper is living through ...
3,1007,TD-VAE,The lottery is an Institution designed to catc...
4,88,Human,You run an RPG pawn shop. You haggle with adve...


### Prompt text

In [ ]:
# Define the evaluation prompt text
#change parts of this prompt according to your technique

PROMPT_TEXT = """
prompt here...

Use the full scale from 1 to 5 as defined, where:
1 = very poor, 2 = poor, 3 = average, 4 = good, 5 = excellent.
Evaluate the story on the following six metrics:

1. Relevance
2. Coherence
3. Empathy
4. Surprise
5. Engagement
6. Complexity

Definitions:
- Relevance: How well the story matches the given prompt.
- Coherence: How logically consistent and well-structured the story is.
- Empathy: The emotional depth of the story and its ability to evoke feelings.
- Surprise: The degree of unexpectedness or originality in the story.
- Engagement: How interesting and captivating the story is.
- Complexity: The richness, depth, and sophistication of the narrative.

Return ONLY a valid JSON object with exactly these keys:
relevance, coherence, empathy, surprise, engagement, complexity, justifications.

Rules:
- Scores must be integers from 1 to 5.
- Do not include any text outside the JSON.
- Do not include explanations outside the "justifications" field.
- The "justifications" field must be an object with exactly these keys:
  relevance, coherence, empathy, surprise, engagement, complexity
- Each justification must be short, at most 1 to 2 sentences.
- Do not include markdown.

Example format:
{
  "relevance": 4,
  "coherence": 3,
  "empathy": 2,
  "surprise": 4,
  "engagement": 3,
  "complexity": 2,
  "justifications": {
    "relevance": "The story follows the prompt closely.",
    "coherence": "Some transitions are unclear.",
    "empathy": "Limited emotional depth.",
    "surprise": "Contains an unexpected twist.",
    "engagement": "Moderately interesting throughout.",
    "complexity": "Simple narrative structure."
  }
}
""".strip()

print("Prompt version:", PROMPT_VERSION)

Prompt version: role_1_prompting


### Prompt builder

In [ ]:
# Build the final prompt sent to a model
def build_prompt(original_prompt: str, story_text: str) -> str:
    return f"""
{PROMPT_TEXT}

Original writing prompt:
{original_prompt}

Story:
{story_text}
""".strip()

### Response schema validation

In [ ]:
# Expected score keys
SCORE_KEYS = [
    "relevance",
    "coherence",
    "empathy",
    "surprise",
    "engagement",
    "complexity"
]

# Parse and validate model output
def parse_and_validate_response(raw_text: str) -> dict:
    if raw_text is None or not str(raw_text).strip():
        raise ValueError("Empty response")

    try:
        data = json.loads(raw_text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", raw_text, re.DOTALL)
        if not match:
            raise ValueError("No JSON object found in response")
        data = json.loads(match.group())

    for key in SCORE_KEYS:
        if key not in data:
            raise ValueError(f"Missing key: {key}")
        if not isinstance(data[key], int):
            raise ValueError(f"Value for {key} is not an integer: {data[key]}")
        if data[key] < 1 or data[key] > 5:
            raise ValueError(f"Value for {key} is out of range 1 to 5: {data[key]}")

    if "justifications" not in data:
        raise ValueError("Missing key: justifications")

    if not isinstance(data["justifications"], dict):
        raise ValueError("The justifications field must be an object")

    for key in SCORE_KEYS:
        if key not in data["justifications"]:
            raise ValueError(f"Missing justification for: {key}")
        if not isinstance(data["justifications"][key], str):
            raise ValueError(f"Justification for {key} is not a string")

    return data

### Shared helper functions

In [ ]:
# Compute mean safely from a list of numeric values
def safe_mean(values):
    clean_values = [v for v in values if v is not None]
    if not clean_values:
        return None
    return mean(clean_values)

# Save progress to CSV
def save_progress(results_list, base_output_dir, filename):
    progress_df = pd.DataFrame(results_list)
    file_path = os.path.join(base_output_dir, filename)
    progress_df.to_csv(file_path, index=False)
    print(f"Saved progress to: {file_path}")

## **GPT**

### GPT API call function

In [ ]:
# Send prompt to GPT model and return raw response
def call_gpt(prompt_text: str):
    response = openai_client.responses.create(
        model=GPT_MODEL,
        input=prompt_text,
        temperature=0
    )

    raw_text = response.output_text

    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "input_tokens", None) if usage else None
    output_tokens = getattr(usage, "output_tokens", None) if usage else None

    return raw_text, input_tokens, output_tokens

### GPT evaluation wrapper

In [ ]:
# Evaluate a story with GPT and return parsed result
def evaluate_with_gpt(prompt_text: str):
    raw_text = None
    input_tokens = None
    output_tokens = None

    try:
        raw_text, input_tokens, output_tokens = call_gpt(prompt_text)
        parsed = parse_and_validate_response(raw_text)

        return {
            "scores": parsed,
            "raw_response": raw_text,
            "parse_success": True,
            "error": None,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens
        }

    except Exception as e:
        return {
            "scores": None,
            "raw_response": raw_text,
            "parse_success": False,
            "error": str(e),
            "input_tokens": input_tokens,
            "output_tokens": output_tokens
        }

## **CLAUDE**

### Claude API call function

In [ ]:
# Send prompt to Claude model and return raw response
def call_claude(prompt_text: str):
    response = anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1000,
        temperature=0,
        messages=[
            {"role": "user", "content": prompt_text}
        ]
    )

    raw_text = response.content[0].text if response.content else None

    input_tokens = getattr(response.usage, "input_tokens", None) if getattr(response, "usage", None) else None
    output_tokens = getattr(response.usage, "output_tokens", None) if getattr(response, "usage", None) else None

    return raw_text, input_tokens, output_tokens

### Claude evaluation wrapper

In [ ]:
# Evaluate a story with Claude and return parsed result
def evaluate_with_claude(prompt_text: str):
    raw_text = None
    input_tokens = None
    output_tokens = None

    try:
        raw_text, input_tokens, output_tokens = call_claude(prompt_text)
        parsed = parse_and_validate_response(raw_text)

        return {
            "scores": parsed,
            "raw_response": raw_text,
            "parse_success": True,
            "error": None,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens
        }

    except Exception as e:
        return {
            "scores": None,
            "raw_response": raw_text,
            "parse_success": False,
            "error": str(e),
            "input_tokens": input_tokens,
            "output_tokens": output_tokens
        }

## **prompt sending**

In [ ]:
# Dispatch evaluation based on model name
def evaluate_with_model(model_name: str, prompt_text: str):
    if model_name == "gpt":
        return evaluate_with_gpt(prompt_text)
    elif model_name == "claude":
        return evaluate_with_claude(prompt_text)
    else:
        raise ValueError(f"Unsupported model name: {model_name}")

In [ ]:
# Evaluate one story with one model multiple times and compute average scores
def evaluate_story_with_repeats(
    model_name: str,
    original_prompt: str,
    story_text: str,
    repeats: int = 3,
    delay_seconds: float = 1.0
):
    prompt_text = build_prompt(original_prompt, story_text)

    raw_responses = []
    run_errors = []
    run_statuses = []
    justification_runs = []

    metric_runs = {
        "relevance": [],
        "coherence": [],
        "empathy": [],
        "surprise": [],
        "engagement": [],
        "complexity": []
    }

    for run_idx in range(repeats):
        result = evaluate_with_model(model_name, prompt_text)

        raw_responses.append(result["raw_response"])
        run_errors.append(result["error"])

        if result["parse_success"] and result["scores"] is not None:
            scores = result["scores"]

            for key in SCORE_KEYS:
                metric_runs[key].append(scores[key])

            justification_runs.append(scores["justifications"])
        else:
            for key in SCORE_KEYS:
                metric_runs[key].append(None)

            justification_runs.append(None)

        run_statuses.append({
            "parse_success": result["parse_success"],
            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"]
        })

        if run_idx < repeats - 1:
            time.sleep(delay_seconds)

    successful_runs = sum(1 for status in run_statuses if status["parse_success"])

    averaged_scores = {
        f"avg_{key}": safe_mean(metric_runs[key])
        for key in SCORE_KEYS
    }

    avg_input_tokens = safe_mean([
        status["input_tokens"] for status in run_statuses
        if status["input_tokens"] is not None
    ])

    avg_output_tokens = safe_mean([
        status["output_tokens"] for status in run_statuses
        if status["output_tokens"] is not None
    ])

    return {
        "successful_runs": successful_runs,
        "relevance_runs": json.dumps(metric_runs["relevance"]),
        "coherence_runs": json.dumps(metric_runs["coherence"]),
        "empathy_runs": json.dumps(metric_runs["empathy"]),
        "surprise_runs": json.dumps(metric_runs["surprise"]),
        "engagement_runs": json.dumps(metric_runs["engagement"]),
        "complexity_runs": json.dumps(metric_runs["complexity"]),
        "justifications_runs": json.dumps(justification_runs, ensure_ascii=False),
        "raw_responses": json.dumps(raw_responses, ensure_ascii=False),
        "run_statuses": json.dumps(run_statuses, ensure_ascii=False),
        "run_errors": json.dumps(run_errors, ensure_ascii=False),
        "avg_input_tokens": avg_input_tokens,
        "avg_output_tokens": avg_output_tokens,
        **averaged_scores
    }

### Test it on one story with GPT

In [ ]:
row = eval_df.iloc[0]

gpt_repeat_test = evaluate_story_with_repeats(
    model_name="gpt",
    original_prompt=row["prompt"],
    story_text=row["story"],
    repeats=REPEATS_PER_MODEL,
    delay_seconds=1.0
)

print(gpt_repeat_test)

{'successful_runs': 3, 'relevance_runs': '[1, 1, 1]', 'coherence_runs': '[1, 1, 1]', 'empathy_runs': '[1, 1, 1]', 'surprise_runs': '[2, 2, 2]', 'engagement_runs': '[1, 1, 1]', 'complexity_runs': '[1, 1, 1]', 'justifications_runs': '[{"relevance": "The story does not address the prompt about a 4-D star colliding with Earth or the resulting new continents and seas.", "coherence": "The narrative is fragmented, repetitive, and lacks clear structure or logical flow.", "empathy": "There is minimal emotional content or connection to characters or events.", "surprise": "Some unusual phrasing and repetition create slight unpredictability, but no meaningful twist.", "engagement": "The story is difficult to follow and does not maintain interest.", "complexity": "The narrative is simplistic and lacks depth or sophistication."}, {"relevance": "The story does not clearly address the prompt about a 4-D star colliding with Earth and the resulting new geography.", "coherence": "The narrative is fragmen

### Test it on one story with Claude

In [ ]:
row = eval_df.iloc[0]

claude_repeat_test = evaluate_story_with_repeats(
    model_name="claude",
    original_prompt=row["prompt"],
    story_text=row["story"],
    repeats=REPEATS_PER_MODEL,
    delay_seconds=1.0
)

print(claude_repeat_test)

{'successful_runs': 3, 'relevance_runs': '[2, 2, 2]', 'coherence_runs': '[2, 2, 2]', 'empathy_runs': '[2, 2, 2]', 'surprise_runs': '[2, 2, 2]', 'engagement_runs': '[2, 2, 2]', 'complexity_runs': '[2, 2, 2]', 'justifications_runs': '[{"relevance": "The story does not clearly address the given prompt about a 4-D star colliding with Earth.", "coherence": "The narrative is disjointed and difficult to follow, with unclear transitions and a lack of logical flow.", "empathy": "The story fails to evoke any strong emotional response or connection with the characters.", "surprise": "There is little to no unexpected or original elements in the story.", "engagement": "The story is not particularly interesting or captivating to read.", "complexity": "The narrative structure is simplistic and lacks depth or sophistication."}, {"relevance": "The story does not clearly address the given prompt about a 4-D star colliding with Earth.", "coherence": "The narrative is disjointed and lacks a clear structur

### full evaluation with GPT

In [ ]:
# Run full evaluation with GPT and save progress
gpt_results = []

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating with GPT"):
    repeat_result = evaluate_story_with_repeats(
        model_name="gpt",
        original_prompt=row["prompt"],
        story_text=row["story"],
        repeats=REPEATS_PER_MODEL,
        delay_seconds=1.0
    )

    result_row = {
        "story_id": row["story_id"],
        "source_model": row["source_model"],
        "original_prompt": row["original_prompt"],
        "human_reference_story": row["human_reference_story"],
        "story": row["story"],

        "human_relevance": row["avg_relevance"],
        "human_coherence": row["avg_coherence"],
        "human_empathy": row["avg_empathy"],
        "human_surprise": row["avg_surprise"],
        "human_engagement": row["avg_engagement"],
        "human_complexity": row["avg_complexity"],

        "judge_model": GPT_MODEL,
        "prompt_version": PROMPT_VERSION,
        "repeats_requested": REPEATS_PER_MODEL,

        **repeat_result
    }

    gpt_results.append(result_row)

    if (idx + 1) % SAVE_EVERY_N_STORIES == 0:
        save_progress(
            gpt_results,
            DRIVE_OUTPUT_DIR,
            f"results_gpt_{PROMPT_VERSION}_progress.csv"
        )

Evaluating with GPT:   0%|          | 0/1056 [00:00<?, ?it/s]

Saved progress to: results/results_gpt_role_1_prompting_progress.csv
Saved progress to: results/results_gpt_role_1_prompting_progress.csv
Saved progress to: results/results_gpt_role_1_prompting_progress.csv
Saved progress to: results/results_gpt_role_1_prompting_progress.csv
Saved progress to: results/results_gpt_role_1_prompting_progress.csv
Saved progress to: results/results_gpt_role_1_prompting_progress.csv
Saved progress to: results/results_gpt_role_1_prompting_progress.csv
Saved progress to: results/results_gpt_role_1_prompting_progress.csv
Saved progress to: results/results_gpt_role_1_prompting_progress.csv
Saved progress to: results/results_gpt_role_1_prompting_progress.csv
Saved progress to: results/results_gpt_role_1_prompting_progress.csv
Saved progress to: results/results_gpt_role_1_prompting_progress.csv
Saved progress to: results/results_gpt_role_1_prompting_progress.csv
Saved progress to: results/results_gpt_role_1_prompting_progress.csv
Saved progress to: results/results

### save final GPT results

In [ ]:
# Save final GPT results
gpt_results_df = pd.DataFrame(gpt_results)
gpt_results_df.to_csv(os.path.join(DRIVE_OUTPUT_DIR, f"results_gpt_{PROMPT_VERSION}.csv"), index=False)

print("GPT results saved")
print(gpt_results_df.shape)
gpt_results_df.head()

GPT results saved
(1056, 33)


,story_id,source_model,original_prompt,human_reference_story,story,human_relevance,human_coherence,human_empathy,human_surprise,human_engagement,...,run_statuses,run_errors,avg_input_tokens,avg_output_tokens,avg_relevance,avg_coherence,avg_empathy,avg_surprise,avg_engagement,avg_complexity
0,0,Human,When you die the afterlife is an arena where y...,"3,000 years have I been fighting. Every mornin...","3,000 years have I been fighting. Every mornin...",3.666667,3.666667,2.333333,2.333333,3.333333,...,"[{""parse_success"": true, ""input_tokens"": 775, ...","[null, null, null]",775,190.666667,5.0,5.000000,4.0,3.000000,4.000000,4.000000
1,1,Human,A new law is enacted that erases soldiers memo...,"“Dad, you 're on TV again !” I heard Eric 's v...","“Dad, you 're on TV again !” I heard Eric 's v...",5.000000,4.666667,4.000000,3.666667,3.666667,...,"[{""parse_success"": true, ""input_tokens"": 811, ...","[null, null, null]",811,161.666667,5.0,4.666667,4.0,3.000000,4.000000,3.666667
2,2,Human,A scientific study proves that all humans have...,"When Tyler entered the ward, his daughter Vale...","When Tyler entered the ward, his daughter Vale...",4.666667,4.666667,4.000000,4.333333,4.000000,...,"[{""parse_success"": true, ""input_tokens"": 1519,...","[null, null, null]",1519,198.666667,5.0,5.000000,5.0,3.666667,4.666667,4.666667
3,3,Human,Write a story about an elderly wizard and his ...,His body was failing. He had taken care of it ...,His body was failing. He had taken care of it ...,3.666667,4.000000,3.000000,2.000000,3.666667,...,"[{""parse_success"": true, ""input_tokens"": 1483,...","[null, null, null]",1483,181.666667,5.0,4.000000,4.0,4.000000,4.333333,4.000000
4,4,Human,"You have become death, destroyer of worlds.","I saw the button. It was simple, red, no words...","I saw the button. It was simple, red, no words...",4.666667,5.000000,3.000000,4.666667,3.666667,...,"[{""parse_success"": true, ""input_tokens"": 671, ...","[null, null, null]",671,182.333333,5.0,4.666667,4.0,3.000000,4.000000,3.666667


### full evaluation with CLAUDE

In [ ]:
# Run full evaluation with Claude and save progress
claude_results = []

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating with Claude"):
    repeat_result = evaluate_story_with_repeats(
        model_name="claude",
        original_prompt=row["prompt"],
        story_text=row["story"],
        repeats=REPEATS_PER_MODEL,
        delay_seconds=1.0
    )

    result_row = {
        "story_id": row["story_id"],
        "source_model": row["source_model"],
        "original_prompt": row["original_prompt"],
        "human_reference_story": row["human_reference_story"],
        "story": row["story"],

        "human_relevance": row["avg_relevance"],
        "human_coherence": row["avg_coherence"],
        "human_empathy": row["avg_empathy"],
        "human_surprise": row["avg_surprise"],
        "human_engagement": row["avg_engagement"],
        "human_complexity": row["avg_complexity"],

        "judge_model": CLAUDE_MODEL,
        "prompt_version": PROMPT_VERSION,
        "repeats_requested": REPEATS_PER_MODEL,

        **repeat_result
    }

    claude_results.append(result_row)

    if (idx + 1) % SAVE_EVERY_N_STORIES == 0:
        save_progress(
            claude_results,
            DRIVE_OUTPUT_DIR,
            f"results_claude_{PROMPT_VERSION}_progress.csv"
        )

Evaluating with Claude:   0%|          | 0/10 [00:00<?, ?it/s]

Saved progress to: results/results_claude_role_1_prompting_progress.csv
Saved progress to: results/results_claude_role_1_prompting_progress.csv


### save final Claude results

In [ ]:
# Save final Claude results
claude_results_df = pd.DataFrame(claude_results)
claude_results_df.to_csv(os.path.join(DRIVE_OUTPUT_DIR, f"results_claude_{PROMPT_VERSION}.csv"), index=False)

print("Claude results saved")
print(claude_results_df.shape)
claude_results_df.head()

Claude results saved
(1056, 33)


,story_id,source_model,original_prompt,human_reference_story,story,human_relevance,human_coherence,human_empathy,human_surprise,human_engagement,...,run_statuses,run_errors,avg_input_tokens,avg_output_tokens,avg_relevance,avg_coherence,avg_empathy,avg_surprise,avg_engagement,avg_complexity
0,0,Human,When you die the afterlife is an arena where y...,"3,000 years have I been fighting. Every mornin...","3,000 years have I been fighting. Every mornin...",3.666667,3.666667,2.333333,2.333333,3.333333,...,"[{""parse_success"": true, ""input_tokens"": 862, ...","[null, null, null]",862,263.333333,5.0,4.000000,4.0,4.000000,5.0,4.0
1,1,Human,A new law is enacted that erases soldiers memo...,"“Dad, you 're on TV again !” I heard Eric 's v...","“Dad, you 're on TV again !” I heard Eric 's v...",5.000000,4.666667,4.000000,3.666667,3.666667,...,"[{""parse_success"": true, ""input_tokens"": 914, ...","[null, null, null]",914,238.333333,4.0,4.000000,4.0,4.000000,4.0,4.0
2,2,Human,A scientific study proves that all humans have...,"When Tyler entered the ward, his daughter Vale...","When Tyler entered the ward, his daughter Vale...",4.666667,4.666667,4.000000,4.333333,4.000000,...,"[{""parse_success"": true, ""input_tokens"": 1739,...","[null, null, null]",1739,250.333333,5.0,4.000000,4.0,4.000000,5.0,4.0
3,3,Human,Write a story about an elderly wizard and his ...,His body was failing. He had taken care of it ...,His body was failing. He had taken care of it ...,3.666667,4.000000,3.000000,2.000000,3.666667,...,"[{""parse_success"": true, ""input_tokens"": 1644,...","[null, null, null]",1644,232.666667,5.0,4.333333,4.0,4.333333,5.0,5.0
4,4,Human,"You have become death, destroyer of worlds.","I saw the button. It was simple, red, no words...","I saw the button. It was simple, red, no words...",4.666667,5.000000,3.000000,4.666667,3.666667,...,"[{""parse_success"": true, ""input_tokens"": 762, ...","[null, null, null]",762,259.000000,5.0,4.666667,4.0,5.000000,5.0,5.0


In [ ]:
# Quick sanity checks
print(gpt_results_df[[
    "story_id", "source_model", "avg_relevance", "avg_coherence",
    "avg_empathy", "avg_surprise", "avg_engagement", "avg_complexity",
    "successful_runs"
]].head())

print(claude_results_df[[
    "story_id", "source_model", "avg_relevance", "avg_coherence",
    "avg_empathy", "avg_surprise", "avg_engagement", "avg_complexity",
    "successful_runs"
]].head())

   story_id source_model  avg_relevance  avg_coherence  avg_empathy  \
0       260         CTRL            1.0       1.000000          1.0   
1       832       Fusion            2.0       2.000000          2.0   
2       846       Fusion            1.0       1.000000          1.0   
3      1007       TD-VAE            2.0       1.666667          1.0   
4        88        Human            5.0       4.000000          4.0   

   avg_surprise  avg_engagement  avg_complexity  successful_runs  
0      2.000000               1        1.000000                3  
1      2.000000               2        2.000000                3  
2      1.333333               1        1.000000                3  
3      2.000000               2        1.666667                3  
4      4.000000               4        4.000000                3  
   story_id source_model  avg_relevance  avg_coherence  avg_empathy  \
0       260         CTRL              2              2            2   
1       832       Fusion     

In [ ]:
print("GPT failed rows:", (gpt_results_df["successful_runs"] == 0).sum())
print("Claude failed rows:", (claude_results_df["successful_runs"] == 0).sum())

GPT failed rows: 0
Claude failed rows: 0
